In [1]:
# Define a function to somewhat crop the data and pack it with wavelength
def reduce(path, wavelength_file):
    
    import os
    from tqdm import tqdm
    
    # Load wavelength data first and close file immediately
    with fits.open(wavelength_file) as wf:
        ll = np.copy(wf[0].data[:800])
    
    # Open data file with context manager to ensure it's closed
    with fits.open(path) as f:
        NY, NStokes, NWavelengths, NX = f[0].data.shape
        
        print("info::old shape is: ", f[0].data.shape)
        
        data_mini = np.zeros([NY,2,800, NX-200])
        
        print("info::new shape is:", data_mini.shape)
        
        for i in tqdm(range(NY)):
            temp = f[0].data[i,:,:800,100:-100]
            
            #print(temp.shape)
            data_mini[i,0,:,:] = temp[0,:,:]
            data_mini[i,1,:,:] = temp[3,:,:]
            del(temp)
    
    data_mini = data_mini.transpose(3,0,1,2)  # Reorder to (NX, NY, 2, 800)
    data_mini = data_mini.astype(np.float32)  # Convert to float32 for smaller file size
    
    kek = fits.PrimaryHDU(data_mini)
    but = fits.ImageHDU(ll)
    lol = fits.HDUList([kek, but])
    filename = os.path.basename(path).replace('.fits', '_reduced.fits')
    lol.writeto(filename, overwrite=True)

In [2]:
import numpy as np 
import matplotlib.pyplot as plt
from astropy.io import fits

In [5]:
reduce('~/data/scratch/scan5.spt.lr0050.lb1.xb1.yb1.demod.xtc.si.ofs.fits', '~/data/scratch/wavelength.fits')

info::old shape is:  (444, 4, 967, 1403)
info::new shape is: (444, 2, 800, 1203)


  0%|          | 0/444 [00:00<?, ?it/s]

100%|██████████| 444/444 [00:16<00:00, 27.48it/s]
